In [ ]:
#@title Mount Drive & unzip dataset
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

DRIVE_BASE = "/content/drive/MyDrive/respiratory_project"
RAW_DIR    = "/content/data/raw"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/logs", exist_ok=True)

# Unzip dataset (only needed first time, takes ~1 min)
zip_path = f"{DRIVE_BASE}/ICBHI_final_database.zip"

# Because the zip contains a folder named "ICBHI_final_database"
extract_check_file = f"{RAW_DIR}/ICBHI_final_database/filename_differences.txt"

if not os.path.exists(extract_check_file):
    print("Unzipping dataset...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(RAW_DIR)
    print("Done.")
else:
    print("Dataset already extracted.")

# Verify
dataset_dir = f"{RAW_DIR}/ICBHI_final_database"
wav_files = [f for f in os.listdir(dataset_dir) if f.endswith(".wav")]

print(f"WAV files found: {len(wav_files)}")  # should be 920

Mounted at /content/drive
Unzipping dataset...
Done.
WAV files found: 920


In [ ]:
#@title Clone your GitHub repo
import os

REPO_URL = "https://github.com/Younes-Abbes/respiratory-classification.git"
REPO_DIR = "/content/respiratory-classification"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Pull latest changes if already cloned
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
print("Repo ready at:", os.getcwd())

/content/respiratory-classification
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.95 KiB | 2.95 MiB/s, done.
From https://github.com/Younes-Abbes/respiratory-classification
   272ebd9..763bfc4  main       -> origin/main
Updating 272ebd9..763bfc4
Fast-forward
 src/train.py | 265 +++++++++++++++++++++++++++++++++++++----------------------
 1 file changed, 165 insertions(+), 100 deletions(-)
/content/respiratory-classification
Repo ready at: /content/respiratory-classification


In [ ]:
#@title install dependencies
%pip install -q -r requirements_colab.txt

In [ ]:
#@title Verify GPU
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU name:     ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("CUDA version: ", torch.version.cuda)

assert torch.cuda.is_available(), "⚠️ No GPU — go to Runtime > Change runtime type > T4 GPU"

GPU available: True
GPU name:      Tesla T4
CUDA version:  12.8


In [ ]:
#@title Link dataset into repo structure
import os


# Symlink dataset
!ln -sf /content/data/raw/ICBHI_final_database /content/respiratory-classification/data/raw

# Verify dataset (920 wav files)
!python scripts/verify_dataset.py

# Generate patient-level splits (train.csv + test.csv)
# Only run if splits don't already exist
if not os.path.exists("data/splits/train.csv"):
    !python scripts/prepare_data.py
else:
    print("Splits already exist — skipping.")

WAV files found: 920
TXT files found: 922
WAV files missing annotation: set()
✅ Dataset verified.
{0: 3642, 1: 1864, 2: 886, 3: 506}
✅ Annotation parser verified.
Splits already exist — skipping.


In [ ]:
#@title WandB setup (experiment tracking)
import wandb
wandb.login()  # paste your API key from wandb.ai/authorize

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: younes-abbes (younes-abbes-insat) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["PYTHONUNBUFFERED"] = "1"

In [ ]:
import torch

model = load_model(use_pretrained=True, freeze_backbone=True).cuda()
dummy = torch.randn(2, 1, 128, 345).cuda()
out = model(dummy)
print("Output shape:", out.shape)  # must be (2, 4)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                            | Status     |                                                                                                
-------------------------------+------------+------------------------------------------------------------------------------------------------
classifier.layernorm.bias      | UNEXPECTED |                                                                                                
classifier.dense.weight        | UNEXPECTED |                                                                                                
classifier.layernorm.weight    | UNEXPECTED |                                                                                                
classifier.dense.bias          | UNEXPECTED |                                                                                                
embeddings.position_embeddings | MISMATCH   | Reinit due to size mismatch ckpt: t

Output shape: torch.Size([2, 4])


In [ ]:
#@title setting dry run
import yaml

config_path = "configs/baseline.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

config["use_pretrained"] = False
config["freeze_backbone"] = True
config["batch_size"] = 2
config["amp"] = True

with open(config_path, "w") as f:
    yaml.dump(config, f)

print("Config reverted to dry-run safe state:")
print(f"  use_pretrained: {config['use_pretrained']}")

Config reverted to dry-run safe state:
  use_pretrained: False


In [ ]:
#@title Dry run
# Confirm GPU + dataset + model all work before committing to full training
!python -m src.train --dry-run

Epoch 0: train_loss=1.2777 val_metrics={'sensitivity': 1.0, 'specificity': 0.0, 'score': 0.5, 'harmonic_score': 0.0}
Epoch 1: train_loss=1.3144 val_metrics={'sensitivity': 1.0, 'specificity': 0.0, 'score': 0.5, 'harmonic_score': 0.0}
Dry run complete, checkpoint saved to checkpoints/dry_run_ckpt.pt
Epoch 0: train_loss=1.3035 val_metrics={'sensitivity': 0.0, 'specificity': 1.0, 'score': 0.5, 'harmonic_score': 0.0}
Epoch 1: train_loss=1.2949 val_metrics={'sensitivity': 0.0, 'specificity': 1.0, 'score': 0.5, 'harmonic_score': 0.0}
Dry run complete, checkpoint saved to checkpoints/dry_run_ckpt.pt


In [ ]:
#@title 7a - Switch to pretrained AST
import yaml

config_path = "configs/baseline.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# Switch to full pretrained AST
config["use_pretrained"] = True
config["freeze_backbone"] = True   # keep frozen for first run
config["batch_size"] = 2           # safe for free tier T4
config["amp"] = True               # mandatory on free tier

with open(config_path, "w") as f:
    yaml.dump(config, f)

print("Config updated:")
print(f"  use_pretrained: {config['use_pretrained']}")
print(f"  freeze_backbone: {config['freeze_backbone']}")
print(f"  batch_size: {config['batch_size']}")

Config updated:
  use_pretrained: True
  freeze_backbone: True
  batch_size: 2


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "src.train", "--dry-run"],
    capture_output=True,
    text=True,
    cwd="/content/respiratory-classification"
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Return code:", result.returncode)

STDOUT: Epoch 0: train_loss=1.3238 val_metrics={'sensitivity': 1.0, 'specificity': 0.0, 'score': 0.5, 'harmonic_score': 0.0}
Epoch 1: train_loss=1.3240 val_metrics={'sensitivity': 1.0, 'specificity': 0.0, 'score': 0.5, 'harmonic_score': 0.0}
Dry run complete, checkpoint saved to checkpoints/dry_run_ckpt.pt
Epoch 0: train_loss=1.2755 val_metrics={'sensitivity': 1.0, 'specificity': 0.0, 'score': 0.5, 'harmonic_score': 0.0}
Epoch 1: train_loss=1.2389 val_metrics={'sensitivity': 1.0, 'specificity': 0.0, 'score': 0.5, 'harmonic_score': 0.0}
Dry run complete, checkpoint saved to checkpoints/dry_run_ckpt.pt

STDERR: 
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2036.12it/s, Materializing param=layernorm.weight]
ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                            | Status     |                                                                                                
-------------------------------+------------+---------------------

In [ ]:
#@title 7b - Full training with checkpoint to Drive
!python -u -m src.train \
    --checkpoint-dir /content/drive/MyDrive/respiratory_project/checkpoints \
    --log-dir /content/drive/MyDrive/respiratory_project/logs

Streaming output truncated to the last 5000 lines.
  batch 1037/2002  loss=0.8308
  batch 1038/2002  loss=0.9080
  batch 1039/2002  loss=1.0201
  batch 1040/2002  loss=1.2997
  batch 1041/2002  loss=1.1461
  batch 1042/2002  loss=0.7068
  batch 1043/2002  loss=1.3593
  batch 1044/2002  loss=1.3491
  batch 1045/2002  loss=1.3390
  batch 1046/2002  loss=1.0644
  batch 1047/2002  loss=1.6559
  batch 1048/2002  loss=0.8045
  batch 1049/2002  loss=1.1746
  batch 1050/2002  loss=1.2006
  batch 1051/2002  loss=0.8881
  batch 1052/2002  loss=0.9046
  batch 1053/2002  loss=1.1362
  batch 1054/2002  loss=1.0964
  batch 1055/2002  loss=1.4878
  batch 1056/2002  loss=1.5552
  batch 1057/2002  loss=1.3140
  batch 1058/2002  loss=1.3228
  batch 1059/2002  loss=0.9293
  batch 1060/2002  loss=1.0575
  batch 1061/2002  loss=0.6391
  batch 1062/2002  loss=1.0008
  batch 1063/2002  loss=1.0143
  batch 1064/2002  loss=0.9770
  batch 1065/2002  loss=1.6173
  batch 1066/2002  loss=1.3738
  batch 1067/2002  

In [ ]:
#@title 7c - Save everything back to Drive
import shutil

# Copy logs and final checkpoint to Drive
shutil.copytree(
    "/content/respiratory-classification/logs",
    "/content/drive/MyDrive/respiratory_project/logs",
    dirs_exist_ok=True
)

print("✅ All results saved to Google Drive.")

✅ All results saved to Google Drive.


In [ ]:
import os, shutil
from pathlib import Path

drive_ckpt = Path("/content/drive/MyDrive/respiratory_project/checkpoints")
drive_logs = Path("/content/drive/MyDrive/respiratory_project/logs")
drive_ckpt.mkdir(parents=True, exist_ok=True)
drive_logs.mkdir(parents=True, exist_ok=True)

# Copy checkpoints
for f in Path("checkpoints").glob("*.pt"):
    shutil.copy(f, drive_ckpt / f.name)
    print(f"✅ Saved: {f.name}")

# Copy logs
for f in Path("logs").glob("*"):
    shutil.copy(f, drive_logs / f.name)
    print(f"✅ Saved: {f.name}")

print("\nAll saved. Safe to close.")

✅ Saved: dry_run_ckpt.pt
✅ Saved: sample_spectrograms.png
✅ Saved: class_distribution.png

All saved. Safe to close.
